# Qwen2.5-1.5B-Instruct Inference
Loading and running the Qwen/Qwen2.5-1.5B-Instruct model using 🤗 Transformers.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer,TrainingArguments
import torch
from  peft import LoraConfig
from trl import SFTConfig,SFTTrainer
model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
print('Tokenizer loaded ✓')
print('Chat template:', tokenizer.chat_template)
# Load model — uses GPU if available, otherwise CPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)
print('Model loaded ✓')
print('Device map:', model.hf_device_map)
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)


In [ ]:
def format_prompts(batch):
    formatted_texts = []
    # Iterate through the batch of examples
    for q, a in zip(batch["question"], batch["answer_heb"]):
        # Create a structured prompt text
        text = f"### Instruction:\n{q}\n\n### Response:\n{a}"
        formatted_texts.append(text)
    
    # Return a dictionary with a single key containing the list of formatted strings
    return {"text": formatted_texts}

In [ ]:
from datasets import load_dataset
TRAINING_PATH="/english_hebrew_questions_200.jsonl"
dataset=load_dataset("json",data_files=TRAINING_PATH,split="train")
split_dataset=dataset.train_test_split(test_size=0.1,seed=42)
formatted_dataset = split_dataset.map(format_prompts,batched=True)
dataset=dataset.shuffle(seed=42)

In [ ]:
rank_dim=8
lora_alpha=5
lora_dropout=0.05

peft_config=LoraConfig(
    r=rank_dim,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules="all-linear",
    bias="none",
    task_type="CASUAL_LM"
)

In [ ]:
training_args=SFTConfig(
    output_dir="./sft_output",
    max_steps=200,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=20,
    eval_strategy="steps",
    eval_steps=50
)

trainer=SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    dataset_text_field="text",
    peft_config=peft_config,
#        processing_class=tokenizer,

)

In [ ]:
trainer.train()

In [ ]:
# merged_model=peft_model.merge_and_unload()